# Lab: From a pattern to a statistic

STAT-S681 · Simulation-Based Inference · Session 10

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ridgecrest_sim import fit, window, fault_centre, offset_from_fault

## Today's question

You can often see that simulated data don't look like the real data. In this lab, you will practice turning what you see into a number that you can formally check.

We will practice on a model of the aftershocks of the July 2019 magnitude 7.1 Ridgecrest,
California earthquake, and a model that has already been fitted.

## The data and the model

The data are 1109 aftershocks (magnitude ≥ 2.5, 0.25–30 days after the
mainshock). Each location is given as $U$, km along the main fault
(positive to the NW), and $V$, km across it (positive to the NE). Plotting
$V$ against $U$ gives a "straightened" map of the fault zone.

The fitted model (details in `10-model-description.html`):

* Most events lie in one band around one gently curved fault, the red
  curve on the maps. How busy the band is varies along the fault, and within
  the band, events are spread across the fault by the same Normal amount
  everywhere.
* A small share of events (about 7%) is more broadly scattered.

We have already simulated 300 replicate catalogs from the fitted model.

In [ ]:
obs = pd.read_csv("data/observed_catalog.csv")           # the real catalog
reps_long = pd.read_csv("data/replicates.csv.gz")        # 300 simulated catalogs, stacked
reps = [d.drop(columns="rep").reset_index(drop=True)     # list of 300 catalogs
        for _, d in reps_long.groupby("rep")]
B = len(reps)
obs.head()

Every catalog, real or simulated, has these columns:

| Column | Meaning |
|---|---|
| `u_km`, `v_km` | position along and across the reference line |
| `offset_km` | distance from the fitted fault curve, NE positive (also `offset_from_fault(cat)`) |
| `t_days`, `mag` | time after the mainshock and magnitude (not needed today) |

## Look first

First plot the observed catalog and three replicates. The red curve is the fitted fault.

In [ ]:
fault_uv = pd.DataFrame(dict(u_km=np.arange(fit["u_lo"], fit["u_hi"], 0.5)))
fault_uv["v_km"] = fault_centre(fault_uv["u_km"])

titles = ["observed"] + [f"replicate {i}" for i in range(1, 4)]
fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True, layout="constrained")
for ax, cat, title in zip(axes, [obs] + reps[:3], titles):
    ax.plot(fault_uv["u_km"], fault_uv["v_km"], color="red")
    ax.scatter(cat["u_km"], cat["v_km"], s=2, alpha=0.5, color="black")
    ax.set_xlim(window["u"])
    ax.set_ylim(window["v"])
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=10)
fig.supxlabel("U: km along the fault (NW positive)")
fig.supylabel("V: km across (NE positive)")
plt.show()

As a warmup, check whether the model seems to get the number of events approximately correct.
The helper function below might help, you would just need to write a function that can be passed as the `statistic` argument.

`check_statistic()` applies any function of one catalog to the observed
catalog and to all 300 replicates, then draws the replicates' values with
the observed value in red.

In [ ]:
def check_statistic(statistic, label="statistic"):
    T_obs = statistic(obs)
    T_rep = np.array([statistic(r) for r in reps])
    print(pd.Series(dict(observed=T_obs, replicate_min=T_rep.min(),
                         replicate_median=np.median(T_rep), replicate_max=T_rep.max(),
                         n_reps_at_or_above=np.sum(T_rep >= T_obs),
                         n_reps_at_or_below=np.sum(T_rep <= T_obs))).round(3).to_string())
    whole = np.all(T_rep == np.round(T_rep))  # counts get one bar per value
    bins = np.arange(T_rep.min() - 0.5, T_rep.max() + 1.5) if whole else 30
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(T_rep, bins=bins, color="grey", edgecolor="white")
    ax.axvline(T_obs, color="red", lw=2)
    ax.set_xlabel(label)
    ax.set_ylabel("replicates")
    ax.set_title("red: observed", loc="left", fontsize=10)
    plt.show()

## Part 1: Describe what you see

With a partner, look at the maps and pick **one** pattern in the observed
catalog that seems unusual compared with the replicates. Check it against
all three replicates, not just one. Describe it in words before you read
on.

1. What is the pattern?

   *Answer:*



## Part 2: Turn the observation into a statistic

1. Complete this sentence for your pattern:

> For every catalog, I will calculate ______. If the discrepancy I noticed is
> real, the observed value should be unusually **high / low**.

   *Answer:*



## Part 3: Implement and compare



### A first attempt: overall clustering

Start with the **mean three-neighbour distance**: for each event, average its
distances to its three nearest neighbours, then average over all events.
Smaller values mean events are more tightly clustered.

You'll first need a function that returns, for each event in a catalog, its
average distance to its three nearest neighbours. Hint: with NumPy you can compute all
pairwise distances at once: with `X = cat[["u_km", "v_km"]].to_numpy()`,
`D = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))`.
Each event's distance to itself is 0; `np.fill_diagonal(D, np.inf)` removes
it, and then `np.sort(D, axis=1)[:, :3]` (or the faster `np.partition`) gives
the three nearest. This takes a few hundredths of a second per catalog: test it
on `obs` first, and consider computing it once for every catalog and storing it
as a new column that later statistics can reuse.
`scipy.spatial.cKDTree(X).query(X, k=4)` and
`sklearn.neighbors.NearestNeighbors` find nearest neighbours even faster (the
first neighbour they return is the event itself).

**Predict before you run:** will the observed value be smaller or larger
than most replicates' values?

*Answer:*

In [ ]:
def mean_neighbor_distance(cat):
    return ...  # for each event: average distance to its 3 nearest neighbours


def statistic(cat):
    return ...  # one number for this catalog


check_statistic(statistic, "mean 3-neighbour distance (km)")

### Your own statistic

Now write a statistic that measures **your** discrepancy from Part 1.

In [ ]:
def statistic2(cat):
    return ...  # one number for this catalog


check_statistic(statistic2, "...")

## Part 4: Interpret


1. What did you find?

   *Answer:*


2. What aspect of the map does your statistic capture? What could still be
   wrong with the model even if your statistic looked typical?

   *Answer:*